In [ ]:
# TensorFlow library for building and training neural networks (GPU supported)
import tensorflow as tf

# NumPy for numerical operations
import numpy as np

# Keras modules to build MLP model
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Flatten

# Dataset for handwritten digits
from tensorflow.keras.datasets import mnist

# Adam optimizer for training
from tensorflow.keras.optimizers import Adam

In [ ]:
# Print available GPUs
# If GPU is listed, TensorFlow will automatically use it
print("Available GPUs:", tf.config.list_physical_devices('GPU'))

Available GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [ ]:
# Load MNIST handwritten digit dataset
(X_train, y_train), (X_test, y_test) = mnist.load_data()

# Normalize pixel values from [0,255] to [0,1]
# Normalization improves convergence speed and stability
X_train = X_train / 255.0
X_test = X_test / 255.0

11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step


In [ ]:
# Learning rates to be tested
learning_rates = [0.01, 0.001]

# Number of neurons in hidden layer
hidden_units = [128, 256]

# Batch sizes for training
batch_sizes = [32, 64]

In [ ]:
# Variables to store best accuracy and best parameters
best_accuracy = 0
best_params = {}

# Explicitly specify GPU usage (TensorFlow will use GPU if available)
with tf.device('/GPU:0'):

    # Loop through all learning rates
    for lr in learning_rates:

        # Loop through all hidden layer sizes
        for units in hidden_units:

            # Loop through all batch sizes
            for batch in batch_sizes:

                # Build MLP model
                model = Sequential([
                    # Flatten converts 28×28 image to 784 input features
                    Flatten(input_shape=(28, 28)),

                    # Hidden layer with ReLU activation
                    Dense(units, activation='relu'),

                    # Output layer with 10 neurons (digits 0–9)
                    Dense(10, activation='softmax')
                ])

                # Compile the model
                model.compile(
                    optimizer=Adam(learning_rate=lr),  # Learning rate from grid
                    loss='sparse_categorical_crossentropy',
                    metrics=['accuracy']
                )

                # Train model
                model.fit(
                    X_train, y_train,
                    epochs=5,                 # Small epochs for grid search
                    batch_size=batch,         # Batch size from grid
                    validation_split=0.2,     # Validation set for CV-like behavior
                    verbose=0                 # Silent training
                )

                # Evaluate model on test data
                _, acc = model.evaluate(X_test, y_test, verbose=0)

                # Print current model performance
                print(f"LR={lr}, Units={units}, Batch={batch} → Accuracy={acc:.4f}")

                # Update best model if accuracy improves
                if acc > best_accuracy:
                    best_accuracy = acc
                    best_params = {
                        'learning_rate': lr,
                        'units': units,
                        'batch_size': batch
                    }

/usr/local/lib/python3.12/dist-packages/keras/src/layers/reshaping/flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


LR=0.01, Units=128, Batch=32 → Accuracy=0.9583
LR=0.01, Units=128, Batch=64 → Accuracy=0.9691
LR=0.01, Units=256, Batch=32 → Accuracy=0.9617
LR=0.01, Units=256, Batch=64 → Accuracy=0.9640
LR=0.001, Units=128, Batch=32 → Accuracy=0.9767
LR=0.001, Units=128, Batch=64 → Accuracy=0.9712
LR=0.001, Units=256, Batch=32 → Accuracy=0.9788
LR=0.001, Units=256, Batch=64 → Accuracy=0.9724


In [ ]:
# Display best hyperparameters obtained from grid search
print("\nBest Accuracy from Grid Search:", best_accuracy)
print("Best Hyperparameters:", best_params)


Best Accuracy from Grid Search: 0.9787999987602234
Best Hyperparameters: {'learning_rate': 0.001, 'units': 256, 'batch_size': 32}


In [ ]:
# Build final model using best hyperparameters
final_model = Sequential([
    Flatten(input_shape=(28, 28)),
    Dense(best_params['units'], activation='relu'),
    Dense(10, activation='softmax')
])

# Compile final model
final_model.compile(
    optimizer=Adam(learning_rate=best_params['learning_rate']),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [ ]:
# Train final model with optimal hyperparameters
history = final_model.fit(
    X_train, y_train,
    epochs=10,                        # More epochs for final training
    batch_size=best_params['batch_size'],
    validation_split=0.2,
    verbose=1
)

Epoch 1/10
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 6s 3ms/step - accuracy: 0.8741 - loss: 0.4294 - val_accuracy: 0.9560 - val_loss: 0.1478
Epoch 2/10
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - accuracy: 0.9670 - loss: 0.1136 - val_accuracy: 0.9722 - val_loss: 0.0961
Epoch 3/10
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - accuracy: 0.9790 - loss: 0.0719 - val_accuracy: 0.9738 - val_loss: 0.0855
Epoch 4/10
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - accuracy: 0.9867 - loss: 0.0472 - val_accuracy: 0.9733 - val_loss: 0.0892
Epoch 5/10
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - accuracy: 0.9891 - loss: 0.0354 - val_accuracy: 0.9745 - val_loss: 0.0841
Epoch 6/10
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - accuracy: 0.9928 - loss: 0.0239 - val_accuracy: 0.9755 - val_loss: 0.0848
Epoch 7/10
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - accuracy: 0.9948 - loss: 0.0174 - val_accuracy: 0.9784 - val_loss: 0.0787
Epoch 8/10
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - accuracy: 0.9964 - loss: 0.0131 - 

In [ ]:
# Evaluate final model on unseen test data
loss, accuracy = final_model.evaluate(X_test, y_test, verbose=0)

# Print final accuracy
print(f"\nFinal Test Accuracy using Best Hyperparameters: {accuracy*100:.2f}%")


Final Test Accuracy using Best Hyperparameters: 97.86%
